# 02 filtering and sampling

Placeholder only. Implementation will be added after the preceding pipeline step has been run and verified.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
import pandas as pd

from src.config import (
    DATASET_ID,
    DATASET_REVISION,
    PILOT_SCAN_LIMIT,
    PILOT_SAMPLE_SIZE,
    RANDOM_SEED,
    SAMPLES_DIR,
)

from src.data_loading import (
    append_experiment_log,
    load_lmsys_stream,
)

from src.filtering import (
    build_relevant_candidate_pool,
    deterministic_sample,
    debug_relevance_matches,
)

In [ ]:
print("Dataset:", DATASET_ID)
print("Revision:", DATASET_REVISION)
print("Conversation scan limit:", PILOT_SCAN_LIMIT)
print("Pilot target:", PILOT_SAMPLE_SIZE)
print("Random seed:", RANDOM_SEED) 

In [ ]:
stream = load_lmsys_stream()

print(stream)

In [ ]:
candidates, diagnostics = build_relevant_candidate_pool(
    dataset=stream,
    max_conversations=PILOT_SCAN_LIMIT,
)

diagnostics

In [ ]:
candidate_df = pd.DataFrame(candidates)

print("Diagnostics:")
print(diagnostics)

print("\nCandidate shape:")
print(candidate_df.shape)

print("\nCategory combinations:")
print(candidate_df["relevance_categories"].value_counts())

print("\nIndividual category counts:")
print(
    candidate_df["relevance_categories"]
    .str.split("|")
    .explode()
    .value_counts()
)

In [ ]:
print("Candidate pairs:", len(candidate_df))

print(
    "Unique source conversations:",
    candidate_df["source_index"].nunique(),
)

print("\nPairs per source conversation:")
print(
    candidate_df["source_index"]
    .value_counts()
    .value_counts()
    .sort_index()
)

print("\nRedacted flag counts:")
print(
    candidate_df["redacted"]
    .value_counts(dropna=False)
)

In [ ]:
pd.set_option("display.max_colwidth", 300)

inspection_df = candidate_df.sample(
    n=min(20, len(candidate_df)),
    random_state=RANDOM_SEED,
)

inspection_df[
    [
        "source_index",
        "pair_index",
        "user_text",
        "assistant_text",
        "relevance_categories",
    ]
]